In [2]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.10.7-hotspot"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("LondonBikes") \
    .master("local[*]") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version: {spark.version}")
print("Running in local mode — all cores available on this machine")

Spark version: 4.1.1
Running in local mode — all cores available on this machine


In [3]:
# %% Cell 2 — Load cleaned Parquet into Spark
df = spark.read.parquet("../data/processed/cleaned.parquet")

print(f"Rows loaded: {df.count()}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

Rows loaded: 5477
Columns: 19
root
 |-- date: timestamp_ntz (nullable = true)
 |-- rides: long (nullable = true)
 |-- temp_mean: double (nullable = true)
 |-- temp_max: double (nullable = true)
 |-- temp_min: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- windspeed: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- weather_desc: string (nullable = true)
 |-- is_holiday: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: long (nullable = true)
 |-- season: long (nullable = true)
 |-- season_name: string (nullable = true)
 |-- is_rainy: long (nullable = true)
 |-- day_type: string (nullable = true)
 |-- rides_vs_yearly_avg: double (nullable = true)



In [4]:
# %% Cell 3 — Spark SQL: average rides by day type
df.createOrReplaceTempView("rides")

print("=== AVERAGE RIDES BY DAY TYPE (Spark SQL) ===")
spark.sql("""
    SELECT
        day_type,
        COUNT(*) AS num_days,
        ROUND(AVG(rides), 0) AS avg_rides,
        ROUND(MIN(rides), 0) AS min_rides,
        ROUND(MAX(rides), 0) AS max_rides
    FROM rides
    GROUP BY day_type
    ORDER BY avg_rides DESC
""").show()

=== AVERAGE RIDES BY DAY TYPE (Spark SQL) ===
+--------+--------+---------+---------+---------+
|day_type|num_days|avg_rides|min_rides|max_rides|
+--------+--------+---------+---------+---------+
| Weekday|    3788|  28077.0|      188|    73094|
| Weekend|    1564|  23445.0|     3531|    70170|
| Holiday|     125|  20545.0|     4327|    67034|
+--------+--------+---------+---------+---------+



In [5]:
# %% Cell 4 — Spark SQL: average rides by season
print("=== AVERAGE RIDES BY SEASON (Spark SQL) ===")
spark.sql("""
    SELECT
        season_name,
        COUNT(*) AS num_days,
        ROUND(AVG(rides), 0) AS avg_rides,
        ROUND(AVG(temp_mean), 1) AS avg_temp_celsius
    FROM rides
    GROUP BY season_name
    ORDER BY avg_rides DESC
""").show()

=== AVERAGE RIDES BY SEASON (Spark SQL) ===
+-----------+--------+---------+----------------+
|season_name|num_days|avg_rides|avg_temp_celsius|
+-----------+--------+---------+----------------+
|     Summer|    1380|  33296.0|            17.4|
|     Autumn|    1363|  27715.0|            11.9|
|     Spring|    1380|  26359.0|             9.9|
|     Winter|    1354|  18828.0|             5.6|
+-----------+--------+---------+----------------+



In [6]:
# %% Cell 5 — Native PySpark API: holiday + rain combined effect
print("=== HOLIDAY + RAIN COMBINED EFFECT (PySpark API) ===")
df.groupBy("is_holiday", "is_rainy") \
  .agg(
      F.count("*").alias("num_days"),
      F.round(F.avg("rides"), 0).alias("avg_rides")
  ) \
  .withColumn("holiday_status",
      F.when(F.col("is_holiday") == 1, "Holiday").otherwise("Non-holiday")
  ) \
  .withColumn("rain_status",
      F.when(F.col("is_rainy") == 1, "Rainy").otherwise("Dry")
  ) \
  .select("holiday_status", "rain_status", "num_days", "avg_rides") \
  .orderBy("avg_rides", ascending=False) \
  .show()

=== HOLIDAY + RAIN COMBINED EFFECT (PySpark API) ===
+--------------+-----------+--------+---------+
|holiday_status|rain_status|num_days|avg_rides|
+--------------+-----------+--------+---------+
|   Non-holiday|        Dry|    3496|  28685.0|
|       Holiday|        Dry|      85|  23442.0|
|   Non-holiday|      Rainy|    1856|  23028.0|
|       Holiday|      Rainy|      40|  14388.0|
+--------------+-----------+--------+---------+



In [10]:
# %% Cell 7 — Filter and describe using PySpark
# Demonstrate Spark's ability to filter and compute stats on the fly

print("=== BUSIEST 10 DAYS EVER (PySpark) ===")
df.select("date", "rides", "temp_mean", "weather_desc", "day_type", "season_name") \
  .orderBy("rides", ascending=False) \
  .show(10)

print("=== SUMMARY STATISTICS FOR RIDES ===")
df.select("rides", "temp_mean", "precipitation", "windspeed").describe().show()

=== BUSIEST 10 DAYS EVER (PySpark) ===
+-------------------+-----+---------+-------------+--------+-----------+
|               date|rides|temp_mean| weather_desc|day_type|season_name|
+-------------------+-----+---------+-------------+--------+-----------+
|2015-07-09 00:00:00|73094|     15.9|Partly cloudy| Weekday|     Summer|
|2020-05-30 00:00:00|70170|     16.4|Partly cloudy| Weekend|     Spring|
|2020-05-25 00:00:00|67034|     17.3|Partly cloudy| Holiday|     Spring|
|2022-08-19 00:00:00|66972|     20.1|      Drizzle| Weekday|     Summer|
|2022-06-21 00:00:00|65326|     16.5|Partly cloudy| Weekday|     Summer|
|2020-06-13 00:00:00|65045|     17.1|         Rain| Weekend|     Summer|
|2020-06-20 00:00:00|64041|     16.3|      Drizzle| Weekend|     Summer|
|2015-08-06 00:00:00|63963|     17.7|      Drizzle| Weekday|     Summer|
|2020-05-31 00:00:00|63116|     17.4|        Clear| Weekend|     Spring|
|2020-06-14 00:00:00|57516|     16.7|      Drizzle| Weekend|     Summer|
+-----------

In [11]:
# %% Cell 8 — Stop Spark session
spark.stop()
print("Spark session stopped. Notebook complete.")

Spark session stopped. Notebook complete.
